# Parallel Processing

Some of the computational tasks performed in scqubits can benefit significantly from parallelization. The scqubits package leverages parallel-processing capabilities provided by the Python Standard Library `multiprocessing` module. For better pickling support, scqubits further supports use of `pathos` and `dill`.

One important consideration for parallelization of tasks like parameter sweeps is the fact that Numpy and Scipy tend to make use of multi-threading internally. (Details of that will depend on how they were built on the machine in question.) This will generally lead to competition between multi-threading on the Numpy/Scipy level and parallelization of `map` methods via `multiprocessing` or `pathos`.

In many cases, best performance is obtained by limiting the number of threads used by Numpy to "a few". (Precise numbers will be machine dependent and need to be determined on a case by case basis.) Limiting this thread number can be achieved from within a Python script or Jupyter and is accomplished by setting environment variables. 

.. note::
    Limiting the number of threads will only be effective if environment variables are set before the first import of
    Numpy. 


Several environment variables can play a role, and which one is needed may again be machine-dependent. 
We thus simply set them all:

In [ ]:
import os

NUM_THREADS = "1"

os.environ["OMP_NUM_THREADS"] = NUM_THREADS
os.environ["OPENBLAS_NUM_THREADS"] = NUM_THREADS
os.environ["MKL_NUM_THREADS"] = NUM_THREADS
os.environ["VECLIB_MAXIMUM_THREADS"] = NUM_THREADS
os.environ["NUMEXPR_NUM_THREADS"] = NUM_THREADS

At this point, Numpy import and import of scqubits can proceed.

In [7]:
import numpy as np

import scqubits
from scqubits import HilbertSpace, InteractionTerm, ParameterSweep

## Enabling parallel processing

Parallel processing is enabled for appropriate scqubits methods and classes by passing the number of cores to be used through the keyword argument `num_cpus`. The following classes and class methods support parallelization:

### Classes and class methods supporting parallelization

| Class or class method                          |
|------------------------------------------------|
| ``ParameterSweep``                             |
| ``HilbertSpace.get_spectrum_vs_paramvals``     |
| ``<qubit_class>.get_spectrum_vs_paramvals``    |
| ``<qubit_class>.plot_evals_vs_paramvals``      |
| ``<qubit_class>.get_matelements_vs_paramvals`` |
| ``<qubit_class>.plot_matelem_vs_paramvals``    |


To use parallelization, the keyword argument `num_cpus` must be passed, specifying the number of cores to be used as an integer, e.g.

In [ ]:
transmon.get_spectrum_vs_paramvals(..., num_cpus=4)

In [ ]:
sweep = ParameterSweep(
    param_name=param_name,
    ...,
    ...,
    num_cpus=4
)

Once `num_cpus` exceeds the value 1 when passed, scqubits starts a parallel processing pool of the desired number of processes.

## When does `num_cpus > 1` actually help?

Parallelization is **not free**, and for many sweeps it gives no speedup — or even a
slowdown. Each grid point is shipped to a worker process (pickling + dispatch), and that
fixed overhead is only worth paying when there is enough work to amortize it:

> `num_cpus > 1` helps only when **(number of grid points) × (cost per point)&nbsp;≫&nbsp;the per-task overhead**.

What to expect, therefore:

- **Small grids, or cheap-per-point systems** (small Hilbert spaces, few eigenstates):
  `num_cpus > 1` gives little or no benefit, and is frequently *slower* than serial. This
  is the common case and is entirely normal — keep the default `num_cpus = 1`.
- **Large grids of expensive points** (large composite Hilbert spaces, many grid points):
  parallel workers pay off.

If a `num_cpus` comparison looks 'inconclusive' or backwards (e.g. `num_cpus=2` slower
than `num_cpus=1`), the usual causes are (i) the sweep is below this break-even, or (ii)
BLAS threads were not capped (see below), so the workers oversubscribe the cores. For a
hands-on demonstration of both regimes, see the `demo_multiprocessing` notebook in the
[scqubits-examples](https://github.com/scqubits/scqubits-examples) repository; to find the
fastest configuration on your machine, use `tools/autotune_multiprocessing.py` from the
source tree.

For large composite systems the per-point **diagonalization method** is often a bigger
lever than parallelism: sparse diagonalization (the default for large spectra; see
`AUTO_SPARSE_DIAG`) can be far faster per point, and once each point is cheap, `num_cpus >
1` helps even less. Try sparse first; parallelize second.


## Global num_cpus default

The global default for `num_cpus` is stored in `scqubits.settings.NUM_CPUS`. Upon import of scqubits, that constant has the value `1` (no parallelization). To change this default and use a user-defined core number by default (say 6), set

In [ ]:
scqubits.settings.NUM_CPUS = 6

## Limiting BLAS threads per worker (`MULTIPROC_BLAS_THREADS`)


As noted at the top of this page, Numpy/Scipy internally multi-thread their linear algebra (through the BLAS backend), which competes with process-level parallelization: with `num_cpus` worker processes each spawning a full BLAS thread pool, the cores become oversubscribed and a sweep can run *slower* than with fewer threads per worker.

Besides exporting the thread-count environment variables before import (shown above), scqubits provides a programmatic cap that is applied automatically while the worker pool is created:


In [ ]:
scqubits.settings.MULTIPROC_BLAS_THREADS = 1   # e.g. 1, or (physical cores) // num_cpus

`MULTIPROC_BLAS_THREADS` accepts a positive integer, or `None` (the default) to leave threading untouched. The cap affects only the worker pool created here; the parent process's environment and BLAS thread count are restored once the pool has been built. Whether it actually limits the workers depends on the platform:

- **Spawn-based workers** (macOS and Windows) re-read the environment when they re-import Numpy/Scipy, so the cap applies directly.
- **Fork-based workers** (Linux) inherit the parent's already-initialized BLAS pool and ignore the environment variables. For these the cap requires the optional [`threadpoolctl`](https://github.com/joblib/threadpoolctl) package (`pip install threadpoolctl`); scqubits then reduces the parent's BLAS thread count for the duration of pool creation so the forked workers inherit it.
- It has **no effect** when Numpy's BLAS exposes no thread control, as with Apple Accelerate on Apple Silicon. Note, however, that Scipy ships its own OpenBLAS there, so the cap still limits the threads used by Scipy's eigensolvers — which is what most scqubits diagonalization relies on.

If the cap cannot take effect on your platform, scqubits emits a one-time warning. The optimal combination of `num_cpus` and thread cap is machine- and workload-dependent, so when performance matters it is worth benchmarking a few values.

## Worker-pool reuse


Within a single computation that issues several parallel `map` calls — for example a `ParameterSweep`, which sweeps each bare subsystem and then the dressed system — scqubits caches the worker pool in `scqubits.settings.POOL` and reuses it whenever the requested core count and backend match, instead of starting a fresh pool each time. The cached pool is shut down automatically at interpreter exit. This is transparent and requires no user action.


## Process start method (`fork` vs `spawn`)

How worker processes are created — the *start method* — is determined by your platform.
There is exactly one safe choice per platform, so scqubits selects it automatically; it is
not a user setting:

| platform | start method | why |
|---|---|---|
| Linux | `fork` | fast, and fork is safe |
| macOS | `spawn` | fork-after-threads is **unsafe** on macOS — Apple's Accelerate/GCD and the Objective-C runtime are not fork-safe, so forking a worker pool after the numerics have started threads can crash, deadlock, or hang. CPython itself defaults macOS to `spawn` since 3.8. This applies to **both Intel and Apple Silicon** Macs. |
| Windows | `spawn` | the only option |

The only consequence you need to be aware of is the `__main__` guard, below.

### The `__main__` guard (`spawn`/`forkserver` only)

With `spawn` (and `forkserver`), each worker process **re-imports your program's entry
module**. A **plain script** that triggers `num_cpus > 1` must therefore guard its entry
point, or the workers would re-run the script and Python raises a `RuntimeError`:

```python
import scqubits as scq

if __name__ == "__main__":
    sweep = scq.ParameterSweep(..., num_cpus=4)
```

**Jupyter/IPython need no guard.** scqubits emits a one-time warning the first time it
starts a `spawn` pool outside IPython, reminding you of this requirement.


### Cost

`spawn` workers re-import numpy/scipy/scqubits, so the **first** parallel sweep of a
session pays a one-time startup of roughly a second. Because the pool is cached and
reused (see *Worker-pool reuse* above), **every subsequent sweep is as fast as fork** — the
cost is paid once per session, not per sweep. For the heavy sweeps where `num_cpus > 1` is
worthwhile, this is negligible.

> **Note:** unlike fork children, `spawn` workers are not automatically reaped if the
> parent process is killed with `SIGKILL` mid-run, and may linger. A normal exit (or a
> `ParameterSweep.run()` completing) cleans them up.


## multiprocessing vs. pathos

scqubits supports parallelization through `multiprocessing` as well as `pathos`. The latter is the default option and is more robust thanks to the advanced pickling methods enabled through `dill`.

To switch from use of `pathos`/`dill` to `multiprocessing`, simply alter the following setting:

In [ ]:
scqubits.settings.MULTIPROC = 'multiprocessing'